<a href="https://colab.research.google.com/github/raeyaanmuppaneni/EMG-controlled-robotic-arm/blob/main/Cross_Validation_Classification_Model_Building.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### Load the CSV files

In [ ]:
# define the paths

# Raw Data
#DATA_CSV = "/content/drive/MyDrive/Technology and Trainings/Python and AI/Raeyaan/Raeyaan shared folder/Exoskeleton Project/Dataset/Raeyaan Dataset/combined_data_EMG_envelop.csv"

# Processed Data
DATA_CSV = "/content/drive/MyDrive/Raeyaan shared folder/Exoskeleton Project/Dataset/Raeyaan Dataset/combined_data_EMG_envelop_20_7.csv"

#Featurized using ts-fresh
#DATA_CSV = "/content/drive/MyDrive/Raeyaan shared folder/Exoskeleton Project/Dataset/Raeyaan Dataset/tsfresh.csv"

#Featurized using librosa/mfcc
#DATA_CSV = "/content/drive/MyDrive/Raeyaan shared folder/Exoskeleton Project/Dataset/Raeyaan Dataset/mfcc.csv"

In [ ]:
import pandas as pd

# load the train csv
data = pd.read_csv(DATA_CSV)

data.head()

,exo_project_emg_1_emg_envelop_0,exo_project_emg_1_emg_envelop_1,exo_project_emg_1_emg_envelop_2,exo_project_emg_1_emg_envelop_3,exo_project_emg_1_emg_envelop_4,exo_project_emg_1_emg_envelop_5,exo_project_emg_1_emg_envelop_6,exo_project_emg_1_emg_envelop_7,exo_project_emg_1_emg_envelop_8,exo_project_emg_1_emg_envelop_9,...,exo_project_emg_1_emg_envelop_51,exo_project_emg_1_emg_envelop_52,exo_project_emg_1_emg_envelop_53,exo_project_emg_1_emg_envelop_54,exo_project_emg_1_emg_envelop_55,exo_project_emg_1_emg_envelop_56,exo_project_emg_1_emg_envelop_57,exo_project_emg_1_emg_envelop_58,exo_project_emg_1_emg_envelop_59,Folder
0,30,30,32,32,32,34,38,38,42,46,...,138,138,148,154,166,180,190,208,208,Hand Lift
1,324,328,330,332,332,334,336,338,342,344,...,410,416,424,424,430,434,434,432,436,Hand Lift
2,356,354,352,350,350,350,348,348,346,346,...,330,328,324,328,328,332,334,338,340,Hand Lift
3,40,40,38,38,38,38,38,38,38,38,...,174,182,184,188,190,190,190,192,196,Hand Lift
4,240,240,240,240,240,240,242,242,246,246,...,384,384,388,388,392,390,390,386,382,Hand Lift


In [ ]:
data["Folder"].unique()

array(['Hand Lift', 'Wrist Lift', 'Hand Twist', 'Normal'], dtype=object)

### Train Test Split

In [ ]:
from sklearn.model_selection import train_test_split

# split the data
train_df, test_df = train_test_split(data, test_size = 0.2,stratify=data['Folder'])

In [ ]:
test_df.to_csv("/content/drive/MyDrive/Raeyaan shared folder/Exoskeleton Project/Dataset/Raeyaan Dataset/raw_EMG_test_20_7.csv",index = False)

In [ ]:
test_df['Folder'].value_counts()

,count
Folder,
Hand Lift,14
Normal,14
Wrist Lift,14
Hand Twist,14


In [ ]:
X = train_df.drop(columns = ["Folder"])
y = train_df["Folder"]

### Encoding Labels

In [ ]:
LABEL_COLUMN = "Folder"

In [ ]:
# labels are there as string otherwise words
# need to convert the labels into numbers
print(train_df[LABEL_COLUMN].value_counts())
LABELS = list(train_df[LABEL_COLUMN].unique())
# sort the labels
LABELS.sort()
print(LABELS.sort())

Folder
Wrist Lift    56
Normal        56
Hand Twist    56
Hand Lift     56
Name: count, dtype: int64
None


In [ ]:
# convert into numbers
train_df[LABEL_COLUMN] = pd.factorize(train_df[LABEL_COLUMN], sort = True)[0]
train_df[LABEL_COLUMN].value_counts()

,count
Folder,
3,56
2,56
1,56
0,56


### Seperate features and labels

In [ ]:
# get all features without the labels
# all the rows
# all the columns without the last column
X = train_df.drop(LABEL_COLUMN, axis = 1).values
# all rows
# only the lastb column, which is the label
Y = train_df[LABEL_COLUMN].values

### Train the Model using Cross Validation

In [ ]:

from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, precision_score, recall_score
import numpy as np

def cross_validation(model, data = (X, Y), splits = 5):
    kf = KFold(n_splits=splits, shuffle=True, random_state=42)

    # Perform k-fold cross-validation
    accuracy = []
    precision = []
    recall = []

    for train_index, valid_index in kf.split(data[0]):
        X_train, X_valid = data[0][train_index], data[0][valid_index]
        y_train, y_valid = data[1][train_index], data[1][valid_index]

        # Fit the defined model
        model.fit(X_train, y_train)

        # Make predictions on the test data
        y_pred = model.predict(X_valid)

        # Calculate accuracy, precision and recall
        accuracy.append(accuracy_score(y_pred, y_valid))
        precision.append(precision_score(y_pred, y_valid, average = 'micro'))
        recall.append(recall_score(y_pred, y_valid, average = 'micro'))


    # get arrays
    accuracy_set = np.array(accuracy)
    precision_set = np.array(precision)
    recall_set = np.array(recall)

    print("Mean Accuracy: {}".format(accuracy_set.mean()))
    print("Mean Precision: {}".format(precision_set.mean()))
    print("Mean Recall: {}".format(recall_set.mean()))
    return accuracy_set.mean()

### Logistic Regression

### KNN

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

# Use 5-fold cross validation for hyper-parameter tuning
# Try out different values and choose the best hyper-parameters
results = []
for i in range(1,15):
  knn = KNeighborsClassifier(n_neighbors=i) # vary n_neighbours from 1 to 20
  result = cross_validation(knn)
  results.append(result)

Mean Accuracy: 0.6524242424242425
Mean Precision: 0.6524242424242425
Mean Recall: 0.6524242424242425
Mean Accuracy: 0.6564646464646465
Mean Precision: 0.6564646464646465
Mean Recall: 0.6564646464646465
Mean Accuracy: 0.6878787878787879
Mean Precision: 0.6878787878787879
Mean Recall: 0.6878787878787879
Mean Accuracy: 0.6565656565656566
Mean Precision: 0.6565656565656566
Mean Recall: 0.6565656565656566
Mean Accuracy: 0.6254545454545454
Mean Precision: 0.6254545454545454
Mean Recall: 0.6254545454545454
Mean Accuracy: 0.6698989898989899
Mean Precision: 0.6698989898989899
Mean Recall: 0.6698989898989899
Mean Accuracy: 0.6565656565656566
Mean Precision: 0.6565656565656566
Mean Recall: 0.6565656565656566
Mean Accuracy: 0.6653535353535354
Mean Precision: 0.6653535353535354
Mean Recall: 0.6653535353535354
Mean Accuracy: 0.6786868686868687
Mean Precision: 0.6786868686868687
Mean Recall: 0.6786868686868687
Mean Accuracy: 0.6831313131313131
Mean Precision: 0.6831313131313131
Mean Recall: 0.6831313

In [ ]:
for i in results:
  print(round(i,2))

0.65
0.66
0.69
0.66
0.63
0.67
0.66
0.67
0.68
0.68
0.69
0.7
0.71
0.7


In [ ]:
from sklearn.neighbors import KNeighborsClassifier

# Use 5-fold cross validation for hyper-parameter tuning
# Try out different values and choose the best hyper-parameters
knn = KNeighborsClassifier(n_neighbors=5) # vary n_neighbours from 1 to 20
model = knn.fit(X_train, y_train)

NameError: name 'X_train' is not defined

NameError: name 'model' is not defined

### Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Use 5-fold cross validation for hyper-parameter tuning
# Try out different values and choose the best hyper-parameters
depth = [2,3,4,5,6,7]
width = [10,20,30,40,50,60,70,80,90,100]
results = []
for i in depth:
  temp = []
  for k in width:
    rf = RandomForestClassifier(n_estimators=k, max_depth = i) # vary n_estimators from 10 to 100 and max_depth from 1 to 10
    result = cross_validation(rf)
    temp.append(result)
  results.append(temp)

Mean Accuracy: 0.6209090909090909
Mean Precision: 0.6209090909090909
Mean Recall: 0.6209090909090909
Mean Accuracy: 0.6658585858585859
Mean Precision: 0.6658585858585859
Mean Recall: 0.6658585858585859
Mean Accuracy: 0.6478787878787878
Mean Precision: 0.6478787878787878
Mean Recall: 0.6478787878787878
Mean Accuracy: 0.6386868686868687
Mean Precision: 0.6386868686868687
Mean Recall: 0.6386868686868687
Mean Accuracy: 0.6476767676767677
Mean Precision: 0.6476767676767677
Mean Recall: 0.6476767676767677
Mean Accuracy: 0.6251515151515151
Mean Precision: 0.6251515151515151
Mean Recall: 0.6251515151515151
Mean Accuracy: 0.6476767676767677
Mean Precision: 0.6476767676767677
Mean Recall: 0.6476767676767677
Mean Accuracy: 0.6477777777777778
Mean Precision: 0.6477777777777778
Mean Recall: 0.6477777777777778
Mean Accuracy: 0.6252525252525252
Mean Precision: 0.6252525252525252
Mean Recall: 0.6252525252525252
Mean Accuracy: 0.6431313131313131
Mean Precision: 0.6431313131313131
Mean Recall: 0.6431313

In [ ]:
rounded_data = [[round(value, 2) for value in row] for row in results]
for i in rounded_data:
  print("\t".join(map(str, i))) # Tab-separated values

0.62	0.67	0.65	0.64	0.65	0.63	0.65	0.65	0.63	0.64
0.71	0.71	0.71	0.7	0.71	0.71	0.71	0.71	0.71	0.71
0.71	0.7	0.71	0.71	0.7	0.71	0.71	0.71	0.72	0.71
0.72	0.71	0.69	0.7	0.7	0.71	0.69	0.7	0.7	0.7
0.72	0.7	0.7	0.71	0.7	0.68	0.68	0.71	0.68	0.7
0.69	0.68	0.68	0.69	0.69	0.68	0.68	0.68	0.68	0.69


##MLP

In [ ]:
from sklearn.neural_network import MLPClassifier

# Use 5-fold cross validation for hyper-parameter tuning
# Try out different values and choose the best hyper-parameters
mlp = MLPClassifier(learning_rate_init=0.0001, max_iter=100) # vary learning rate as 0.0001, 0.001, 0.01, 0.05, 0.1, 1 and max_iter from 10 to 100

cross_validation(mlp)

Mean Accuracy: 0.2006060606060606
Mean Precision: 0.2006060606060606
Mean Recall: 0.2006060606060606


np.float64(0.2006060606060606)

In [ ]:
from sklearn.neural_network import MLPClassifier
import numpy as np
  # Use 5-fold cross validation for hyper-parameter tuning
  # Try out different values and choose the best hyper-parameters
lr = [0.05, 0.01, 0.005, 0.001, 0.0005, 0.0001]
epochs = [20,40,60,80,100,120,140,160,180,200]
results = []

for i in lr:
  temp = []
  for j in epochs:
    mlp = MLPClassifier(hidden_layer_sizes=(200,100,50,), learning_rate_init=i, max_iter=j) # vary learning rate as 0.0001, 0.001, 0.01, 0.05, 0.1, 1 and max_iter from 10 to 100
    print(f"learning rate {i}, epochs {j}")
    result = cross_validation(mlp)
    temp.append(result)
  results.append(temp)

learning rate 0.05, epochs 20


/usr/local/lib/python3.11/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (20) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (20) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (20) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (20) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/neural_network/_multilayer_perceptron.py

Mean Accuracy: 0.2413131313131313
Mean Precision: 0.2413131313131313
Mean Recall: 0.2413131313131313
learning rate 0.05, epochs 40


/usr/local/lib/python3.11/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (40) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (40) reached and the optimization hasn't converged yet.
  warnings.warn(


Mean Accuracy: 0.2011111111111111
Mean Precision: 0.2011111111111111
Mean Recall: 0.2011111111111111
learning rate 0.05, epochs 60


...

/usr/local/lib/python3.11/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (180) reached and the optimization hasn't converged yet.
  warnings.warn(


Mean Accuracy: 0.2588888888888889
Mean Precision: 0.2588888888888889
Mean Recall: 0.2588888888888889
learning rate 0.0001, epochs 200


/usr/local/lib/python3.11/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


Mean Accuracy: 0.34797979797979794
Mean Precision: 0.34797979797979794
Mean Recall: 0.34797979797979794


/usr/local/lib/python3.11/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


In [ ]:
#print(results)
rounded_data = [[round(value, 2) for value in row] for row in results]
for i in rounded_data:
  print("\t".join(map(str, i))) # Tab-separated values

0.24	0.2	0.25	0.25	0.24	0.19	0.28	0.24	0.25	0.2
0.24	0.34	0.38	0.47	0.47	0.53	0.56	0.48	0.53	0.54
0.34	0.25	0.29	0.36	0.44	0.29	0.34	0.43	0.35	0.38
0.26	0.28	0.48	0.47	0.34	0.5	0.51	0.3	0.52	0.35
0.23	0.32	0.43	0.39	0.35	0.44	0.48	0.42	0.45	0.49
0.21	0.21	0.29	0.32	0.33	0.34	0.34	0.28	0.26	0.35


### Fit the best model

- Select the best model having the highest cross validation accuracy

In [ ]:
# Model with the lowest RMSE or highest r2
best_model = RandomForestClassifier(n_estimators=10, max_depth = 5)


# Fit the model on the full training dataset
best_model.fit(X, Y)

RandomForestClassifier(max_depth=5, n_estimators=10)

In [ ]:
import pickle
# Load the trained model
with open("/content/drive/MyDrive/Raeyaan shared folder/Exoskeleton Project/models/randomForest_model_19_7.pkl", "wb") as model_file:
  pickle.dump(best_model, model_file)

### Save the best model

- Make sure to donwload the model when you are planning to use the model later.